# 06 — Relearn task 6–7, dữ liệu (a): cutout, bảng camera và độ phủ các mặt của 50 SKU

Ngày tạo: 31/10/2026. Nhánh `feat/task6-relearn`. Theo [plan](../../docs/project/task6-relearn-plan-2026-10-30.md), Task 9, quyết định Q5: mỗi SKU đủ các mặt RPC đã chụp, dùng góc thấy nhãn, phân bố cân đối.

**Làm gì:** kiểm kê hai bộ cutout mới của 50 SKU relearn; tóm tắt kết quả duyệt (`reviews.json`) và bảng camera (`skus.json`); chốt catalog cutout (M0) gồm 6 bộ v1 cộng 2 bộ mới; kiểm cổng độ phủ Q5 cho task 6–7.

**Không chạy lại trong notebook:** giải nén 1.985 ảnh cam2 còn thiếu từ zip Kaggle; cắt SAM 2.1 base trên GPU (`--per-class 200 --camera-weights 0:1,1:0,2:1,3:0`, log `runs/data_preprocessing/synthetic_relearn/extract.log`); cắt lại bằng SAM 2.1 large 1.086 ảnh cam0 bị SAM base loại vì iou < 0,9 (`resegment.py`, ngưỡng 0,75, log `recut_l.log`); duyệt bằng mắt trên trang lưới (`cutouts/`, `cutouts_l/`, danh sách loại `rejects_b.txt`, `rejects_l.txt`, ghép bằng `build_reviews.py`). Các bước đó chạy ngày 31/10, ở đây chỉ kiểm hash.

**Quyết định của người dùng (chat 31/10):** xác nhận gallery cutout bị loại; chốt bảng camera; chọn phương án A cho 7 SKU trượt cổng khe cam0 (ghi ngoại lệ có lý do trong `configs/data/synth_relearn.json`, giữ luật M0 loại góc hộp lộ hai mặt).

**Input:** `data/archive` (train2019, instances), `data/processed/cutouts_relearn_v1`, `data/processed/cutouts_relearn_l_v1`, catalog v1 `data/processed/rpc_synth_v2_v1/catalog`. **Output:** `data/processed/rpc_synth_relearn_v1/catalog/{cutouts_relearn_v1,cutouts_relearn_l_v1,cutouts,m0_review,coverage}.json`, run receipt ở `runs/data_preprocessing/synthetic_relearn/<run_id>/`.

In [1]:
import json, os, platform, subprocess, sys, uuid, hashlib, time
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists())
os.chdir(REPO); sys.path.insert(0, str(REPO))
TZ = ZoneInfo("Asia/Bangkok")
RUN_ID = uuid.uuid4().hex[:8]
STARTED_AT = datetime.now(TZ).isoformat(timespec="seconds")
RUN_DIR = REPO / "runs/data_preprocessing/synthetic_relearn" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
WORK = REPO / "runs/data_preprocessing/synthetic_relearn"
V1 = Path("data/processed/rpc_synth_v2_v1/catalog")
CAT = Path("data/processed/rpc_synth_relearn_v1/catalog")
CONFIG = "configs/data/synth_relearn.json"
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
ENV = {"run_id": RUN_ID, "started_at": STARTED_AT, "git_revision": git("rev-parse", "HEAD"),
       "git_dirty": bool(git("status", "--porcelain", "--", "autocheckout", "tools", "tests")),
       "python": sys.version.split()[0], "host": platform.node()}
print(json.dumps(ENV, indent=1))

{
 "run_id": "ecab3dfc",
 "started_at": "2026-10-31T12:46:12+07:00",
 "git_revision": "abb86d883953079e339c8cb25bc710172533e4d8",
 "git_dirty": false,
 "python": "3.10.22",
 "host": "archlinux"
}


## 1. Hai bộ cutout mới và kiểm kê (Step 1–3)

`relearn_v1`: SAM 2.1 base, mọi ảnh cam0 và cam2 của 50 SKU (5.438 ảnh, loại tự động khi iou < 0,9 hoặc fill ngoài 0,3–0,98). `relearn_l_v1`: SAM 2.1 large cắt lại các ảnh cam0 bị loại vì iou. Gần hết các bước cam0 thiếu (1.036/1.038) là do loại iou chứ không do RPC thiếu ảnh; lô thử 60 ảnh cho thấy SAM-L không nâng iou dự đoán (trung vị 0,869) nhưng mask nhìn sạch, nên ngưỡng hạ xuống 0,75 và chất lượng do vòng duyệt bằng mắt quyết định. Kiểm kê: file tồn tại, alpha khác rỗng, ảnh nguồn thuộc train2019, SHA-256.

In [2]:
def run(*args):
    t = time.time()
    r = subprocess.run([sys.executable, "-m", "tools.synth_v2", *args], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-2000:]); assert r.returncode == 0, r.stderr[-2000:]
    return round(time.time() - t, 1)

SETS = {"relearn_v1": "data/processed/cutouts_relearn_v1", "relearn_l_v1": "data/processed/cutouts_relearn_l_v1"}
for name, d in SETS.items():
    run("inventory", "--index", f"{d}/index.json", "--cutout-set", name, "--root", "data/archive",
        "--out", str(CAT / f"cutouts_{name}.json"))
    assert not json.load(open(CAT / f"cutouts_{name}.json"))["issues"]
    rejected = json.load(open(f"{d}/rejected.json"))
    print(" ", name, "rejected at extraction:", dict(Counter(r["reason"] for r in rejected)))
for log in ("extract.log", "recut_l.log"):
    print(log, sha(WORK / log)[:16], (WORK / log).read_text().strip().splitlines()[-2])

3568 usable cutouts, 0 issues, cameras {0: 1632, 2: 1936} -> data/processed/rpc_synth_relearn_v1/catalog/cutouts_relearn_v1.json
  relearn_v1 rejected at extraction: {'iou': 1834, 'fill': 36}


1020 usable cutouts, 0 issues, cameras {0: 1020} -> data/processed/rpc_synth_relearn_v1/catalog/cutouts_relearn_l_v1.json
  relearn_l_v1 rejected at extraction: {'iou': 66}
extract.log 9081142a23fbfba2 CANH BAO: 37 lop co it hon 100 cutout: [33, 34, 35, 36, 73, 74, 79, 80, 82, 83, 90, 91, 92, 93, 97, 98, 102, 108, 109, 110]... -> xem rejected.json
recut_l.log a9df15f0eea88d50 1086 targets: 1020 ok, 66 rejected, 513s


## 2. Kết quả duyệt (Step 4)

`reviews.json` = 6.625 review của v1 + review của cutout mới. Cutout mới có ảnh nguồn mà v1 đã giữ (`keep`) bị loại theo luật (`duplicate_source`, mỗi annotation chỉ giữ một cutout). Phần còn lại duyệt bằng mắt theo tiêu chí M0: đĩa xoay hoặc mảnh đĩa trong mask, mask rách, sai vật, góc hộp lộ hai mặt (≥ ~30% bề rộng). 511 cutout SAM-L cam0 của 15 SKU chỉ dùng cam2 không được duyệt: generator không dùng chúng và `finalize` không giữ cutout chưa duyệt. Người dùng xác nhận gallery cutout bị loại (`reject_gallery/`) ngày 31/10.

In [3]:
reviews = json.load(open(CAT / "reviews.json"))
new_ids = {c["cutout_id"] for n in SETS for c in json.load(open(CAT / f"cutouts_{n}.json"))["cutouts"]}
new = [r for r in reviews["reviews"] if r["cutout_id"] in new_ids]
print("reviews", len(reviews["reviews"]), "| new", len(new), "| unreviewed new", len(new_ids) - len(new))
print(Counter((r["reviewer"], r["decision"]) for r in new))
print(Counter(r["reason"] for r in new if r["decision"] == "reject"))
print(reviews["status"])
for p in ("rejects_b.txt", "rejects_l.txt", "build_reviews.py"):
    print(p, sha(WORK / p)[:16])
print("reviews.json", sha(CAT / "reviews.json")[:16])

reviews 10702 | new 4077 | unreviewed new 511
Counter({('claude', 'keep'): 2216, ('rule', 'reject'): 1647, ('claude', 'reject'): 214})
Counter({'duplicate_source': 1647, 'box view shows two faces': 140, 'turntable disc in mask': 32, 'small turntable fragment': 24, 'broken/torn mask': 16, 'wrong object / broken mask': 2})
relearn review done 31/10; reject gallery confirmed by the user (chat 31/10/2026)
rejects_b.txt 934d7ce392e382cb
rejects_l.txt 77ab56af44c4dec0
build_reviews.py fc5270e15a2f992f
reviews.json 264847cfbbd4d6c9


## 3. Bảng camera (Step 5)

Bản sao `skus.json` v1, xét lại 50 SKU trên trang `sheets/` (ảnh val | cam2 | cam0) theo Q5(b): góc nào thấy nhãn thì dùng. Người dùng chốt ngày 31/10: chỉ 119 (flat → mixed: val nửa ngửa nắp, nửa nằm nghiêng lộ nhãn) và 121 (mixed 0,3/0,7 → upright: val gần như toàn nằm nghiêng, nay có 40 cutout cam0) đổi; 123/130 giữ flat (val chỉ thấy mặt trên có nhãn, cam0 là cạnh mỏng), 134/135 giữ mixed.

In [4]:
from autocheckout.taskcfg import TaskConfig

old = {s["rpc_category_id"]: s for s in json.load(open(V1 / "skus.json"))["skus"]}
skus = json.load(open(CAT / "skus.json"))["skus"]
changed = [s for s in skus if s["camera_weights"] != old[s["rpc_category_id"]]["camera_weights"]]
for s in changed:
    o = old[s["rpc_category_id"]]
    print(s["name"], o["shape_group"], o["camera_weights"], "->", s["shape_group"], s["camera_weights"], "|", s["accepted_by"])
task_config = TaskConfig.load(json.load(open(CONFIG))["task_config"])
relearn = {c.rpc_category_id for t in (6, 7) for c in task_config.task(t).classes}
print("shape groups of the 50:", Counter(s["shape_group"] for s in skus if s["rpc_category_id"] in relearn))
print("skus.json", sha(CAT / "skus.json")[:16])

119_canned_food flat {'2': 1.0} -> mixed {'0': 0.5, '2': 0.5} | user 31/10 (AskUserQuestion)
121_canned_food mixed {'0': 0.3, '2': 0.7} -> upright {'0': 0.99, '2': 0.01} | user 31/10 (AskUserQuestion)
shape groups of the 50: Counter({'upright': 27, 'flat': 17, 'mixed': 6})
skus.json 4a1bd9551b5405d5


## 4. Chốt M0 và cổng độ phủ Q5 (Step 6)

Catalog cuối = cutout `keep` trên camera mà SKU dùng, từ 6 bộ v1 và 2 bộ mới. Cổng Q5: mọi kiểu đặt × camera có cutout, và khe xoay cam0 lớn nhất ≤ 2 bước (18°). 7 SKU trượt được người dùng chấp nhận làm ngoại lệ (phương án A, lý do ghi trong config): 38, 97, 98, 102 là hộp, khe nằm ở các góc vì luật M0 loại góc lộ hai mặt nhưng cả bốn mặt đều có ảnh; 90 và 198 khe 27° do mask rách; 112 do RPC không có ảnh cam0 bước 1–2.

In [5]:
v1_sets = ["sam_v1", "cam0_v1", "cam2s_v1", "cam2s_v2", "cam0s_v2", "repl_v1"]
catalogs = [str(V1 / f"cutouts_{n}.json") for n in v1_sets] + [str(CAT / f"cutouts_{n}.json") for n in SETS]
run("finalize", "--catalogs", *catalogs, "--reviews", str(CAT / "reviews.json"), "--skus", str(CAT / "skus.json"),
    "--out-dir", str(CAT), "--accepted-by", "user (chat 31/10/2026)",
    "--notes", "relearn: v1 catalog + SAM-base and SAM-L camera-0 cuts of the 50 SKUs; reject gallery and camera table "
               "confirmed by the user 31/10; coverage exceptions 38 90 97 98 102 112 198 accepted (option A)")
final = json.load(open(CAT / "cutouts.json"))["cutouts"]
per_sku = Counter(c["rpc_category_id"] for c in final)
print("final cutouts", len(final), "by set", dict(Counter(c["cutout_set"] for c in final)))
print("min per SKU: all", min(per_sku.values()), "| the 50", min(per_sku[r] for r in relearn))
run("coverage", "--config", CONFIG, "--tasks", "6", "7")
cov = json.load(open(CAT / "coverage.json"))
for rpc, reason in cov["exceptions"].items():
    print(rpc, cov["skus"][rpc]["cam0_gap"], "-", reason)

M0: 7406 cutouts, min 15/SKU -> data/processed/rpc_synth_relearn_v1/catalog
final cutouts 7406 by set {'cam0_v1': 1369, 'cam0s_v2': 2, 'cam2s_v1': 118, 'cam2s_v2': 926, 'relearn_l_v1': 445, 'relearn_v1': 1084, 'repl_v1': 184, 'sam_v1': 3278}
min per SKU: all 15 | the 50 40


43/50 SKUs cover every placement, camera and angle, 7 excepted -> data/processed/rpc_synth_relearn_v1/catalog/coverage.json
38 {'back': 9, 'front': 7} - box: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
90 {'front': 3} - flat glass bottle: torn camera-0 masks rejected, widest gap 3 steps (27 degrees)
97 {'front': 5} - milk carton: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
98 {'front': 3} - milk carton: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
102 {'front': 3} - milk carton: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
112 {'front': 3} - RPC train2019 has no camera-0 photo at steps 1-2 (gap 3 steps)
198 {'front': 3} - glue stick: torn camera-0 masks rejected, widest gap 3 steps (27 degrees)


In [6]:
FINISHED_AT = datetime.now(TZ).isoformat(timespec="seconds")
summary = {**ENV, "finished_at": FINISHED_AT, "status": "PASS_M0_Q5",
           "outputs": {p: sha(CAT / p) for p in ("cutouts.json", "m0_review.json", "coverage.json", "skus.json", "reviews.json",
                                                 "cutouts_relearn_v1.json", "cutouts_relearn_l_v1.json")},
           "config_sha256": sha(CONFIG),
           "limits": ["visual review by one reviewer (Claude) with user confirmation of the reject gallery",
                      "camera-0 SAM-L cutouts kept from predicted iou 0.75, judged by eye only",
                      "7 SKUs pass the Q5 gate as user-accepted exceptions",
                      "SAM extraction, the Kaggle unzip and the visual review are not re-run here"]}
json.dump(summary, open(RUN_DIR / "summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

{
 "run_id": "ecab3dfc",
 "started_at": "2026-10-31T12:46:12+07:00",
 "git_revision": "abb86d883953079e339c8cb25bc710172533e4d8",
 "git_dirty": false,
 "python": "3.10.22",
 "host": "archlinux",
 "finished_at": "2026-10-31T12:46:33+07:00",
 "status": "PASS_M0_Q5",
 "outputs": {
  "cutouts.json": "5d9335188c31bb06f1befc3e0630b66194b448e1be9eedba9670b49f882b267f",
  "m0_review.json": "18ece652dd7b3652af32d9a2657559ca2d08bf47822294196c6a2ad80f1992bf",
  "coverage.json": "5da9ae050aae799e46e46e22c60b2f03500ceba2d70115e1a00e88f3ac513731",
  "skus.json": "4a1bd9551b5405d5eb4a594497abe0d250d2ea2131ae8fb885470e9d17d502c3",
  "reviews.json": "264847cfbbd4d6c9dd609483a67bf880934b3ab1f017ba150397476a8d555632",
  "cutouts_relearn_v1.json": "aaa43de1ee063109d961ff2ebcbed4905344b6d7a3759a078314d779adcb515e",
  "cutouts_relearn_l_v1.json": "d4d5fb8977b6ee231449816cbed9363357ce50a437389a260ec67082f86c205d"
 },
 "config_sha256": "2611d9bc8660c6c980fee47ec130d000da04733d52441c1e8fc37eafe680884e",
 "limi

## 5. Release học lại task 6–7 (Task 10, Step 1–2)

`configs/data/synth_relearn.json` = `synth_v2.json` với seed 20261031, task config relearn, catalog và `skus.json` ở trên, 12.330 ảnh mỗi task, `view_balance` (mặt và góc dùng đều) và quota mỗi SKU ±5%. Chạy ngoài notebook ngày 31/10 (generate mất 46 phút): `scales`, `plan --tasks 6 7`, `generate --workers 16 --tasks 6 7`, `export`, `verify` (giải mã RLE mọi vật, không lấy mẫu; 35 phút). Log ở `runs/data_preprocessing/synthetic_relearn/`.

**Một cảnh không xếp được.** Lượt generate đầu ghi 24.659/24.660 ảnh: ảnh 6010143 (task 6, hard) gồm 7 túi đồ khô lớn; tổng diện tích val của rổ là 1,27, dưới trần 1,29 của mức hard, nhưng ở tỉ lệ đã fit các bbox chiếm 4,3M px trên khay 3,4M px, nên 32 lần thử đều hỏng. Commit `d7d1d6f` thêm một lượt thử cuối với mọi vật nhỏ đi exp(−σ_jitter) ≈ 0,92 (vẫn trong dải jitter của release); lượt này chỉ chạy khi mọi lần thử trước đã hỏng, nên 24.659 ảnh đã ghi giữ nguyên. Chạy lại generate chỉ render ảnh còn thiếu. Người dùng chọn cách này (phương án A, chat 31/10).

Mục 5–7 chạy trong kernel riêng: không chạy lại mục 1–4 vì `finalize` ghi lại `catalog/cutouts.json` (trường `created_at`), mà plan receipt của release ghim sha của file đó.

In [1]:
import json, os, platform, subprocess, sys, uuid, hashlib
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists())
os.chdir(REPO); sys.path.insert(0, str(REPO))
TZ = ZoneInfo("Asia/Bangkok")
RUN_ID = uuid.uuid4().hex[:8]
STARTED_AT = datetime.now(TZ).isoformat(timespec="seconds")
RUN_DIR = REPO / "runs/data_preprocessing/synthetic_relearn" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT = Path("data/processed/rpc_synth_relearn_v1")
CAT, CONFIG = OUT / "catalog", "configs/data/synth_relearn.json"
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
ENV = {"run_id": RUN_ID, "started_at": STARTED_AT, "git_revision": git("rev-parse", "HEAD"),
       "git_dirty": bool(git("status", "--porcelain", "--", "autocheckout", "tools", "tests")),
       "python": sys.version.split()[0], "host": platform.node()}
print(json.dumps(ENV, indent=1))

receipt = json.load(open(OUT / "plans/plan_receipt.json"))
assert receipt["config_sha256"] == sha(CONFIG) and receipt["inputs"]["catalog"] == sha(CAT / "cutouts.json")
assert receipt["inputs"]["skus"] == sha(CAT / "skus.json")
for g in sorted((OUT / "runs").glob("generate_*.json")):
    r = json.load(open(g))
    print(g.name, "scenes", r["n"], "| failures", r["failures"], "|", r["seconds"], "s")
manifest = json.load(open(OUT / "manifest.json"))
assert manifest["config_sha256"] == sha(CONFIG)
for t, v in manifest["tasks"].items():
    print(f"task {t}: {v['images']} images, {v['objects']} objects, {v['labeled']} labeled")
report = json.load(open(OUT / "verify_report.json"))
print("verify passed", report["passed"], f"({sum(report['gates'].values())}/{len(report['gates'])} gates)", report["counts"])
assert report["passed"] and report["gates"]["sku_quota"] and report["gates"]["view_balance"]
for name, v in report["distributions"].items():
    print(" ", name, json.dumps(v))
label = json.load(open(OUT / "labels/task_6/6010143.json"))
print("scene 6010143:", {k: label["resolved"][k] for k in ("retry", "min_visible")},
      "min visible", min(a["visible_fraction"] for a in label["annotations"]))

{
 "run_id": "556f25ca",
 "started_at": "2026-10-31T14:35:08+07:00",
 "git_revision": "d7d1d6ffaaf692ddce90b8f750864627231da968",
 "git_dirty": false,
 "python": "3.10.22",
 "host": "archlinux"
}
generate_2026-10-31T124806+0700.json scenes 24660 | failures {'6010143': 'image 6010143: no layout after 32 retries'} | 2780.2 s
generate_2026-10-31T134005+0700.json scenes 24660 | failures {} | 17.6 s
task 6: 12330 images, 150971 objects, 75496 labeled
task 7: 12330 images, 150856 objects, 75512 labeled
verify passed True (15/15 gates) {'6': {'images': 12330, 'objects': 150971, 'new_objects': 75496, 'new_share': 0.5001}, '7': {'images': 12330, 'objects': 150856, 'new_objects': 75512, 'new_share': 0.5006}}
  objects_per_image {"medium": {"synth_mean": 12.37, "val_mean": 12.38}, "hard": {"synth_mean": 17.27, "val_mean": 17.29}, "easy": {"synth_mean": 7.07, "val_mean": 7.15}}
  bbox_covered_p50 {"medium": {"synth": 0.343, "val": 0.4}, "hard": {"synth": 0.397, "val": 0.462}, "easy": {"synth": 0.2

In [2]:
cut = {c["cutout_id"]: c["rpc_category_id"] for c in json.load(open(CAT / "cutouts.json"))["cutouts"]}
R1 = Path("data/processed/rpc_synth_v2_v1")
cut_r1 = {c["cutout_id"]: c["rpc_category_id"] for c in json.load(open(R1 / "catalog/cutouts.json"))["cutouts"]}
r1 = Counter()
for t in range(1, 6):
    r1.update(cut_r1[a["cutout_id"]] for a in json.load(open(R1 / f"annotations/task_{t}_rpc.json"))["annotations"]
              if a["labeled"])
v = sorted(r1.values())
print(f"round 1, 200 SKUs: labeled per SKU min/median/max {v[0]}/{v[len(v) // 2]}/{v[-1]}")
for t in (6, 7):
    anns = json.load(open(OUT / f"annotations/task_{t}_rpc.json"))["annotations"]
    lab = Counter(cut[a["cutout_id"]] for a in anns if a["labeled"])
    v, b = sorted(lab.values()), sorted(r1[s] for s in lab)
    print(f"task {t}: {len(lab)} SKUs, labeled per SKU {v[0]}/{v[len(v) // 2]}/{v[-1]} "
          f"(round 1, same SKUs: {b[0]}/{b[len(b) // 2]}/{b[-1]}), new-object share {len([a for a in anns if a['labeled']]) / len(anns):.3f}")

round 1, 200 SKUs: labeled per SKU min/median/max 2663/2897/3282


task 6: 25 SKUs, labeled per SKU 3019/3020/3021 (round 1, same SKUs: 2663/2899/3281), new-object share 0.500


task 7: 25 SKUs, labeled per SKU 3020/3020/3022 (round 1, same SKUs: 2663/2897/3281), new-object share 0.501


## 6. Phủ mặt so với ảnh quầy thật (Step 3, không chặn)

`runs/analysis_r1/d7_faces.py` (thêm `--release`, `--tasks`, `--tag`) so mỗi ảnh val của một SKU với mọi ảnh RPC của SKU đó bằng backbone ConvNeXt V2 trên GPU; một ảnh val tính là "chưa phủ" khi ảnh RPC gần nhất thuộc mặt/góc mà release không dùng và giống hơn ảnh đã dùng tốt nhất ít nhất 0,02. Chạy ngoài notebook (GPU, 2 phút); ở đây đọc `d7_faces.json` (round 1, task 1–5) và `d7_faces_relearn.json` (task 6–7) cho 50 SKU.

In [3]:
before = {r["label"]: r for r in json.load(open("runs/analysis_r1/d7_faces.json"))}
after = {r["label"]: r for r in json.load(open("runs/analysis_r1/d7_faces_relearn.json"))}
print(f"{'label':>5} {'name':24s} {'shape':8s} {'uncovered':>14s} {'s_used_med':>14s}")
for label in sorted(after):
    b, a = before[label], after[label]
    print(f"{label:5d} {a['name'][:24]:24s} {a['shape']:8s} {b['uncovered']:5.3f} -> {a['uncovered']:5.3f} "
          f"{b['s_used_med']:5.3f} -> {a['s_used_med']:5.3f}")
ub, ua = [before[k]["uncovered"] for k in after], [after[k]["uncovered"] for k in after]
sb, sa = [before[k]["s_used_med"] for k in after], [after[k]["s_used_med"] for k in after]
FACES = {"uncovered_mean": [round(float(np.mean(ub)), 3), round(float(np.mean(ua)), 3)],
         "skus_uncovered_ge_25pct": [sum(x >= .25 for x in ub), sum(x >= .25 for x in ua)],
         "s_used_med_mean": [round(float(np.mean(sb)), 3), round(float(np.mean(sa)), 3)],
         "better_same_worse": [sum(y < x for x, y in zip(ub, ua)), sum(y == x for x, y in zip(ub, ua)),
                               sum(y > x for x, y in zip(ub, ua))]}
print(len(after), "SKUs, round 1 -> relearn:", FACES)

label name                     shape         uncovered     s_used_med
   11 23_dried_food            flat     0.075 -> 0.100 0.839 -> 0.834
   17 33_instant_drink         upright  0.025 -> 0.150 0.786 -> 0.781
   18 36_instant_drink         upright  0.200 -> 0.225 0.727 -> 0.737
   19 38_instant_drink         mixed    0.450 -> 0.125 0.668 -> 0.735
   39 80_alcohol               upright  0.300 -> 0.250 0.663 -> 0.686
   41 83_drink                 upright  0.325 -> 0.325 0.717 -> 0.710
   46 91_alcohol               upright  0.250 -> 0.075 0.743 -> 0.739
   49 102_milk                 upright  0.600 -> 0.175 0.703 -> 0.713
   54 108_canned_food          upright  0.375 -> 0.225 0.721 -> 0.735
   55 110_canned_food          upright  0.350 -> 0.275 0.744 -> 0.762
   57 114_canned_food          upright  0.250 -> 0.150 0.693 -> 0.731
   58 116_canned_food          upright  0.375 -> 0.325 0.606 -> 0.648
   59 117_canned_food          upright  0.375 -> 0.200 0.706 -> 0.707
   60 121_canned_foo

## 7. Gallery và gói dữ liệu (Step 4–6)

Gallery `runs/data_preprocessing/synthetic_relearn/gallery/`: 20 ảnh mỗi task có khung (xanh = vật mới có nhãn, đỏ = vật cũ), ba trang synth | val theo mức, và ảnh 6010143 sau khi thu nhỏ. Người dùng chọn phương án A cho ảnh 6010143 ngày 31/10 và không nhận xét riêng về gallery.

Gói `data/packages/relearn_rpc_synth_relearn_v1_<ngày>.tar` (`scripts/pack_relearn_data.sh`): `images`, `tasks`, `manifest.json`, `task_config.json`, `verify_report.json` của release; từ seed `data/training/full-syn-r1`: `plan.json`, `stages/`, model task 5 (`task_final.pth`, `pred_val.npz`, `run_info.json`) và `EXP-FULL-SYN-FSA/task_1/hf_model`. Eval set, ảnh val/test và pretrained không đóng gói lại: máy vast mới cần giải nén thêm gói `full_syn_rpc_synth_v2_v1_<ngày>.tar` của lần M5. Đã giải nén thử ở repo root và `sha256sum -c` (24.679 file OK) ngày 31/10.

In [4]:
PKG = sorted(Path("data/packages").glob("relearn_rpc_synth_relearn_v1_*.tar"))[-1]
RECEIPTS = PKG.with_suffix("")
print((RECEIPTS / "PACKAGE.txt").read_text().strip())
md5 = hashlib.md5()
with open(PKG, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        md5.update(block)
assert md5.hexdigest() == (RECEIPTS / "tar.md5").read_text().strip()
listed = dict(line.split("  ", 1)[::-1] for line in (RECEIPTS / "SHA256SUMS").read_text().splitlines())
for p in ("manifest.json", "verify_report.json", "task_config.json"):
    assert listed[str(OUT / p)] == sha(OUT / p), p
print(PKG.name, f"{PKG.stat().st_size / 2**30:.2f} GiB", "md5", md5.hexdigest(), "|", len(listed), "files in SHA256SUMS")

relearn_rpc_synth_relearn_v1_20261031: 24679 files, 6.8G, git d7d1d6ffaaf692ddce90b8f750864627231da968


relearn_rpc_synth_relearn_v1_20261031.tar 6.73 GiB md5 d4330b997cd31e52da4df36b4f4e3e14 | 24679 files in SHA256SUMS


In [5]:
FINISHED_AT = datetime.now(TZ).isoformat(timespec="seconds")
summary = {**ENV, "finished_at": FINISHED_AT, "status": "PASS_RELEASE",
           "outputs": {p: sha(OUT / p) for p in ("manifest.json", "verify_report.json", "plans/plan_receipt.json")},
           "config_sha256": sha(CONFIG), "verify_gates": f"{sum(report['gates'].values())}/{len(report['gates'])}",
           "faces_round1_to_relearn": FACES,
           "package": {"file": PKG.name, "md5": md5.hexdigest(), "files": len(listed)},
           "limits": ["scales, plan, generate, export, verify, d7_faces and the test extraction are not re-run here",
                      "image 6010143 rendered by the last-resort pass, objects exp(-sigma) smaller (user: option A)",
                      "gallery reviewed by Claude; the user chose option A and did not comment on the gallery separately",
                      "123/130_chocolate keep most val crops on views the release does not use (camera table: flat)"]}
json.dump(summary, open(RUN_DIR / "summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

{
 "run_id": "556f25ca",
 "started_at": "2026-10-31T14:35:08+07:00",
 "git_revision": "d7d1d6ffaaf692ddce90b8f750864627231da968",
 "git_dirty": false,
 "python": "3.10.22",
 "host": "archlinux",
 "finished_at": "2026-10-31T14:35:24+07:00",
 "status": "PASS_RELEASE",
 "outputs": {
  "manifest.json": "94f2ed61eb9ee67674f4eee67b8523a2d717944e498c9ceaef2121f0a7e0d8dc",
  "verify_report.json": "4a29ec6e9d643f166cb4958c516b6cc0d5759a0215a038c3f2087c907234597f",
  "plans/plan_receipt.json": "05ecbc480c32f9834e10d3d05030ee1c4283924681a591985849f1db3f8431b3"
 },
 "config_sha256": "2611d9bc8660c6c980fee47ec130d000da04733d52441c1e8fc37eafe680884e",
 "verify_gates": "15/15",
 "faces_round1_to_relearn": {
  "uncovered_mean": [
   0.265,
   0.16
  ],
  "skus_uncovered_ge_25pct": [
   23,
   10
  ],
  "s_used_med_mean": [
   0.702,
   0.719
  ],
  "better_same_worse": [
   33,
   10,
   7
  ]
 },
 "package": {
  "file": "relearn_rpc_synth_relearn_v1_20261031.tar",
  "md5": "d4330b997cd31e52da4df36b4f